In [ ]:
# Set up the data.
# Reads scalar field data from a full anvil surface and matches it to damaged and undamaged coordinates.
# Iterates across subfolders named after anvil IDs.
# For damaged coordinates: nearest neighbour match from full anvil (1 row per coordinate).
# For undamaged coordinates: all full anvil points within radius_mm are included (more rows than input).
# Output: one semicolon-separated .txt file per anvil.

from pathlib import Path
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

# Paths
root_path  = Path("PATH")   # folder containing one subfolder per anvil

roughness_col = "Roughness (10)"
curvature_col = "Mean curvature (10)"
radius_mm     = 5.64

# Helpers
def load_txt_with_header(path):
    with open(path, "r") as f:
        lines = f.readlines()
    for i, line in enumerate(lines):
        if line.startswith("//"):
            col_names = [c.strip() for c in line.lstrip("/").strip().split(";")]
            df = pd.read_csv(path, sep=";", skiprows=i + 1, header=None, names=col_names)
            df.columns = [c.lstrip("/") for c in df.columns]
            return df
    raise ValueError(f"No '//' header found in {path}")

def load_txt_no_header(path):
    df = pd.read_csv(path, sep=",", header=None, names=["X", "Y", "Z"])
    return df

def save_txt(df, path):
    with open(path, "w") as f:
        f.write("//" + ";".join(df.columns) + "\n")
        df.to_csv(f, index=False, header=False, sep=";")

# Process each anvil subfolder
anvil_dirs = sorted([d for d in root_path.iterdir() if d.is_dir()])
print(f"Found {len(anvil_dirs)} anvil folder(s)")

for anvil_dir in anvil_dirs:
    anvil_id = anvil_dir.name
    print(f"\n{'='*60}")
    print(f"Processing: {anvil_id}")
    print(f"{'='*60}")

    txt_files = list(anvil_dir.glob("*.txt"))

    # Identify files by keyword
    full_anvil_files = [f for f in txt_files if "spatial_join" in f.name.lower()]
    damaged_files    = [f for f in txt_files if "damage" in f.name.lower() and "undamage" not in f.name.lower()]
    undamaged_files  = [f for f in txt_files if "undamage" in f.name.lower()]

    # Validate
    if not full_anvil_files:
        print(f"  Skipping — no spatial_join file found")
        continue
    if not damaged_files:
        print(f"  Skipping — no damaged file found")
        continue
    if not undamaged_files:
        print(f"  Skipping — no undamaged file found")
        continue

    full_anvil_txt = full_anvil_files[0]
    damaged_txt    = damaged_files[0]
    undamaged_txt  = undamaged_files[0]

    print(f"  Full anvil:  {full_anvil_txt.name}")
    print(f"  Damaged:     {damaged_txt.name}")
    print(f"  Undamaged:   {undamaged_txt.name}")

    # Load files
    print("  Loading files...")
    df_full      = load_txt_with_header(full_anvil_txt)
    df_damaged   = load_txt_with_header(damaged_txt)
    df_undamaged = load_txt_no_header(undamaged_txt)

    print(f"    Full anvil:  {len(df_full):,} points")
    print(f"    Damaged:     {len(df_damaged):,} points")
    print(f"    Undamaged:   {len(df_undamaged):,} points")

    # Verify scalar columns exist in full anvil
    for col in [roughness_col, curvature_col]:
        if col not in df_full.columns:
            print(f"  Skipping — '{col}' not found in full anvil. Available: {list(df_full.columns)}")
            continue

    # Build KD-tree on full anvil XYZ
    print("  Building KD-tree on full anvil...")
    full_xyz = df_full[["X", "Y", "Z"]].to_numpy(dtype=float)
    tree = cKDTree(full_xyz)

    # Damaged — nearest neighbour match (1 row per coordinate)
    print("  Matching damaged coordinates (nearest neighbour)...")
    damaged_xyz = df_damaged[["X", "Y", "Z"]].to_numpy(dtype=float)
    _, idx = tree.query(damaged_xyz, k=1)

    df_damaged_out = df_damaged[["X", "Y", "Z"]].copy()
    df_damaged_out[roughness_col] = df_full[roughness_col].values[idx]
    df_damaged_out[curvature_col] = df_full[curvature_col].values[idx]
    df_damaged_out["condition"]   = "damaged"
    print(f"    Output rows: {len(df_damaged_out):,}")

    # Undamaged — all full anvil points within radius_mm
    print(f"  Matching undamaged coordinates (radius query, r={radius_mm}mm)...")
    undamaged_xyz = df_undamaged[["X", "Y", "Z"]].to_numpy(dtype=float)
    neighbours = tree.query_ball_point(undamaged_xyz, r=radius_mm)

    records = []
    for i, neighbour_idx in enumerate(neighbours):
        if len(neighbour_idx) == 0:
            print(f"    Warning: no full anvil points within {radius_mm}mm of undamaged point {i} — skipping")
            continue
        for j in neighbour_idx:
            records.append({
                "X":           full_xyz[j, 0],
                "Y":           full_xyz[j, 1],
                "Z":           full_xyz[j, 2],
                roughness_col: df_full[roughness_col].values[j],
                curvature_col: df_full[curvature_col].values[j],
                "condition":   "undamaged",
            })

    df_undamaged_out = pd.DataFrame.from_records(records)
    print(f"    Input rows:  {len(df_undamaged):,}")
    print(f"    Output rows: {len(df_undamaged_out):,} ({len(df_undamaged_out) / len(df_undamaged):.1f}x expansion)")

    # Combine and save
    print("  Combining and saving...")
    df_out = pd.concat([df_damaged_out, df_undamaged_out], ignore_index=True)
    print(f"    Total output rows: {len(df_out):,}")

    output_path = anvil_dir / f"{anvil_id}_output.txt"
    save_txt(df_out, output_path)
    print(f"    Saved to: {output_path}")

print(f"\nAll done.")

In [ ]:
# Roughness: KDE with scaled number of handled vertices per anvil & condition (damage vs. undamaged).

import os
from pathlib import Path
import math

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Path to the folder that contains one folder per anvil
root_path = Path("PATH")

random_seed = 4

# Load the data
records = []

for anvil_dir in sorted(root_path.iterdir()):
    if not anvil_dir.is_dir():
        continue

    anvil_id = anvil_dir.name

    output_files = list(anvil_dir.glob("*_output.txt"))
    if not output_files:
        print(f"Skipping {anvil_id} — no output.txt found")
        continue

    output_file = output_files[0]
    print(f"Loading {anvil_id}...")

    with open(output_file, "r") as f:
        lines = f.readlines()
    for i, line in enumerate(lines):
        if line.startswith("//"):
            col_names = [c.strip() for c in line.lstrip("/").strip().split(";")]
            df_file = pd.read_csv(output_file, sep=";", skiprows=i + 1, header=None, names=col_names)
            df_file.columns = [c.lstrip("/") for c in df_file.columns]
            break

    if "Roughness (10)" not in df_file.columns or "condition" not in df_file.columns:
        print(f"  Skipping {anvil_id} — missing expected columns")
        continue

    subset = df_file[["Roughness (10)", "Mean curvature (10)", "condition"]].dropna()
    subset = subset.rename(columns={"Roughness (10)": "roughness", "Mean curvature (10)": "curvature"})
    subset["anvil_id"] = anvil_id
    records.append(subset)
    print(f"  Loaded {len(subset):,} rows")

print("\nCombining all anvils...")
df = pd.concat(records, ignore_index=True)
if df.empty:
    raise RuntimeError("No roughness data loaded. Check paths, filenames, and column names.")

print(f"Total rows loaded: {len(df):,}")

print("\nAnvils and conditions found:")
for anvil in sorted(df["anvil_id"].unique()):
    conditions = df[df["anvil_id"] == anvil]["condition"].unique()
    print(f"  {anvil}: {', '.join(conditions)}")

# Balance sample sizes per anvil & condition for plotting
print("\nBalancing sample sizes...")
balanced_chunks = []

for anvil in df["anvil_id"].unique():
    sub_anvil = df[df["anvil_id"] == anvil]

    n_d = (sub_anvil["condition"] == "damaged").sum()
    n_u = (sub_anvil["condition"] == "undamaged").sum()

    if n_d == 0 or n_u == 0:
        balanced_chunks.append(sub_anvil)
        continue

    target_n = min(n_d, n_u)

    damaged_sub = sub_anvil[sub_anvil["condition"] == "damaged"]
    if len(damaged_sub) > target_n:
        damaged_sub = damaged_sub.sample(n=target_n, random_state=random_seed, replace=False)

    undamaged_sub = sub_anvil[sub_anvil["condition"] == "undamaged"]
    if len(undamaged_sub) > target_n:
        undamaged_sub = undamaged_sub.sample(n=target_n, random_state=random_seed, replace=False)

    balanced_chunks.append(pd.concat([damaged_sub, undamaged_sub], ignore_index=True))
    print(f"  {anvil}: n_damaged={n_d:,}, n_undamaged={n_u:,} → balanced to {target_n:,} each")

df_plot = pd.concat(balanced_chunks, ignore_index=True)
print(f"\nBalanced dataset: {len(df_plot):,} total rows")

count_table = (
    df.groupby(["anvil_id", "condition"])
      .size()
      .unstack(fill_value=0)
)

# Create panel plot
print("\nPlotting...")
sns.set_theme(style="whitegrid", context="talk")

anvil_order = [
    # KBN, sandstone
    "KBN-AV2", "KBN-AV3", "KBN-AV4", "KBN-AV5", "KBN-AV6",
    # KBY, sandstone
    "KBY-AV7", "KBY-AV8", "KBY-AV9", "KBY-AV11", "KBY-AV12",
    # AL - High quality limestone/type I
    "AL-AV2", "AL-AV3", "AL-AV5_east", "AL-AV5_west", "AL-AV8",
    # AL - Low quality limestone/type II
    "AL-AV7", "AL-AV9", "AL-AV12", "AL-AV13",
]
anvils = [a for a in anvil_order if a in df_plot["anvil_id"].unique()]
n_anvils = len(anvils)

n_cols = 5
n_rows = 4

fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(22, 12),
    sharex=False,
    sharey=False,
)
axes = axes.flatten()

condition_colors = {
    "undamaged": "#1D9E75",
    "damaged":   "#D85A30",
}

for i, anvil in enumerate(anvils):
    ax = axes[i]
    grp = df_plot[df_plot["anvil_id"] == anvil]

    for condition in ["undamaged", "damaged"]:
        sub = grp[grp["condition"] == condition]
        if sub.empty:
            continue
        sns.kdeplot(
            data=sub, x="roughness", ax=ax,
            label=condition, fill=True, alpha=0.3,
            linewidth=1.5, color=condition_colors.get(condition),
        )

    ax.set_title(anvil)
    ax.set_xlim(-0.1, 2.5)
    ax.set_ylabel("")
    ax.set_xlabel("")

    damaged_n   = len(grp[grp["condition"] == "damaged"])
    undamaged_n = len(grp[grp["condition"] == "undamaged"])
    info_text   = f"n damaged = {damaged_n:,}   |   n undamaged = {undamaged_n:,}"
    ax.text(0.5, -0.12, info_text, transform=ax.transAxes,
            ha="center", va="top", fontsize=7)

for ax in axes[:len(anvils)]:
    ax.tick_params(axis='x', pad=20)

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

fig.text(0.5, 0.04, 'Roughness [mm]', ha='center', fontsize=16)
fig.text(0.04, 0.5, 'Density', va='center', rotation='vertical', fontsize=16)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles, labels,
    loc="center left", bbox_to_anchor=(0.92, 0.5),
    frameon=True, fancybox=True, shadow=True, fontsize=12
)

plt.tight_layout(rect=[0.06, 0.06, 0.90, 0.98])
print("Done.")
plt.show()

In [ ]:
# Curvature: KDE with scaled number of handled vertices per anvil & condition (damage vs. undamaged).

import os
from pathlib import Path
import math

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Path to the folder that contains one folder per anvil
root_path = Path("PATH")

random_seed = 4

# Load the data
records = []

for anvil_dir in sorted(root_path.iterdir()):
    if not anvil_dir.is_dir():
        continue

    anvil_id = anvil_dir.name

    output_files = list(anvil_dir.glob("*_output.txt"))
    if not output_files:
        print(f"Skipping {anvil_id} — no output.txt found")
        continue

    output_file = output_files[0]
    print(f"Loading {anvil_id}...")

    with open(output_file, "r") as f:
        lines = f.readlines()
    for i, line in enumerate(lines):
        if line.startswith("//"):
            col_names = [c.strip() for c in line.lstrip("/").strip().split(";")]
            df_file = pd.read_csv(output_file, sep=";", skiprows=i + 1, header=None, names=col_names)
            df_file.columns = [c.lstrip("/") for c in df_file.columns]
            break

    if "Mean curvature (10)" not in df_file.columns or "condition" not in df_file.columns:
        print(f"  Skipping {anvil_id} — missing expected columns")
        continue

print("\nCombining all anvils...")
records = []

for anvil_dir in sorted(root_path.iterdir()):
    if not anvil_dir.is_dir():
        continue

    anvil_id = anvil_dir.name
    output_files = list(anvil_dir.glob("*_output.txt"))
    if not output_files:
        continue

    output_file = output_files[0]
    with open(output_file, "r") as f:
        lines = f.readlines()

    df_file = None
    for i, line in enumerate(lines):
        if line.startswith("//"):
            col_names = [c.strip() for c in line.lstrip("/").strip().split(";")]
            df_file = pd.read_csv(output_file, sep=";", skiprows=i + 1, header=None, names=col_names)
            df_file.columns = [c.lstrip("/") for c in df_file.columns]
            break

    if df_file is None:
        continue

    if "Mean curvature (10)" not in df_file.columns or "condition" not in df_file.columns:
        continue

    df_file = df_file.rename(columns={"Mean curvature (10)": "curvature", "Roughness (10)": "roughness"})
    if "roughness" not in df_file.columns:
        continue

    df_file["anvil_id"] = anvil_id
    df_file = df_file[["roughness", "curvature", "condition", "anvil_id"]]
    records.append(df_file)

df = pd.concat(records, ignore_index=True) if records else pd.DataFrame(
    columns=["roughness", "curvature", "condition", "anvil_id"]
)
if df.empty:
    raise RuntimeError("No curvature data loaded. Check paths, filenames, and column names.")

print(f"Total rows loaded: {len(df):,}")

print("\nAnvils and conditions found:")
for anvil in sorted(df["anvil_id"].unique()):
    conditions = df[df["anvil_id"] == anvil]["condition"].unique()
    print(f"  {anvil}: {', '.join(conditions)}")

# Balance sample sizes per anvil & condition for plotting
print("\nBalancing sample sizes...")
balanced_chunks = []

for anvil in df["anvil_id"].unique():
    sub_anvil = df[df["anvil_id"] == anvil]

    n_d = (sub_anvil["condition"] == "damaged").sum()
    n_u = (sub_anvil["condition"] == "undamaged").sum()

    if n_d == 0 or n_u == 0:
        balanced_chunks.append(sub_anvil)
        continue

    target_n = min(n_d, n_u)

    damaged_sub = sub_anvil[sub_anvil["condition"] == "damaged"]
    if len(damaged_sub) > target_n:
        damaged_sub = damaged_sub.sample(n=target_n, random_state=random_seed, replace=False)

    undamaged_sub = sub_anvil[sub_anvil["condition"] == "undamaged"]
    if len(undamaged_sub) > target_n:
        undamaged_sub = undamaged_sub.sample(n=target_n, random_state=random_seed, replace=False)

    balanced_chunks.append(pd.concat([damaged_sub, undamaged_sub], ignore_index=True))
    print(f"  {anvil}: n_damaged={n_d:,}, n_undamaged={n_u:,} → balanced to {target_n:,} each")

df_plot = pd.concat(balanced_chunks, ignore_index=True)
print(f"\nBalanced dataset: {len(df_plot):,} total rows")

count_table = (
    df.groupby(["anvil_id", "condition"])
      .size()
      .unstack(fill_value=0)
)

# Create panel plot
print("\nPlotting...")
sns.set_theme(style="whitegrid", context="talk")

anvil_order = [
    # KBN, sandstone
    "KBN-AV2", "KBN-AV3", "KBN-AV4", "KBN-AV5", "KBN-AV6",
    # KBY, sandstone
    "KBY-AV7", "KBY-AV8", "KBY-AV9", "KBY-AV11", "KBY-AV12",
    # AL - High quality limestone/type I
    "AL-AV2", "AL-AV3", "AL-AV5_east", "AL-AV5_west", "AL-AV8",
    # AL - Low quality limestone/type II
    "AL-AV7", "AL-AV9", "AL-AV12", "AL-AV13",
]
anvils = [a for a in anvil_order if a in df_plot["anvil_id"].unique()]
n_anvils = len(anvils)

n_cols = 5
n_rows = 4

fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(22, 12),
    sharex=False,
    sharey=False,
)
axes = axes.flatten()

condition_colors = {
    "undamaged": "#1D9E75",
    "damaged":   "#D85A30",
}

for i, anvil in enumerate(anvils):
    ax = axes[i]
    grp = df_plot[df_plot["anvil_id"] == anvil]

    for condition in ["undamaged", "damaged"]:
        sub = grp[grp["condition"] == condition]
        if sub.empty:
            continue
        sns.kdeplot(
            data=sub, x="curvature", ax=ax,
            label=condition, fill=True, alpha=0.3,
            linewidth=1.5, color=condition_colors.get(condition),
        )

    ax.set_title(anvil)
    ax.set_xlim(-0.01, 0.08)
    ax.set_ylabel("")
    ax.set_xlabel("")

    damaged_n   = len(grp[grp["condition"] == "damaged"])
    undamaged_n = len(grp[grp["condition"] == "undamaged"])
    info_text   = f"n damaged = {damaged_n:,}   |   n undamaged = {undamaged_n:,}"
    ax.text(0.5, -0.12, info_text, transform=ax.transAxes,
            ha="center", va="top", fontsize=7)

for ax in axes[:len(anvils)]:
    ax.tick_params(axis='x', pad=20)

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

fig.text(0.5, 0.04, 'Mean curvature [mm^-1]', ha='center', fontsize=16)
fig.text(0.04, 0.5, 'Density', va='center', rotation='vertical', fontsize=16)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles, labels,
    loc="center left", bbox_to_anchor=(0.92, 0.5),
    frameon=True, fancybox=True, shadow=True, fontsize=12
)

plt.tight_layout(rect=[0.06, 0.06, 0.90, 0.98])
print("Done.")
plt.show()

In [6]:
# Summary table for mean, mode and standard deviation per anvil and condition (roughness and curvature).

summary_records = []

for anvil in sorted(df_plot["anvil_id"].unique()):
    sub = df_plot[df_plot["anvil_id"] == anvil]

    for condition in ["damaged", "undamaged"]:
        cond_sub = sub[sub["condition"] == condition]
        if cond_sub.empty:
            continue

        rough = cond_sub["roughness"]
        curv  = cond_sub["curvature"]

        summary_records.append({
            "anvil_id":         anvil,
            "condition":        condition,
            "mean_roughness":   rough.mean(),
            "std_roughness":    rough.std(),
            "median_roughness": rough.median(),
            "mode_roughness":   rough.mode().iloc[0],
            "min_roughness":    rough.min(),
            "max_roughness":    rough.max(),
            "mean_curvature":   curv.mean(),
            "std_curvature":    curv.std(),
            "median_curvature": curv.median(),
            "mode_curvature":   curv.mode().iloc[0],
            "min_curvature":    curv.min(),
            "max_curvature":    curv.max(),
        })

summary_df = pd.DataFrame.from_records(summary_records)

In [ ]:
# Roughness: Boxplot by locality and raw material.

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

root_path = Path("FILEPATH")

random_seed = 4

# Load data
records = []
for anvil_dir in sorted(root_path.iterdir()):
    if not anvil_dir.is_dir():
        continue
    anvil_id = anvil_dir.name
    output_files = list(anvil_dir.glob("*_output.txt"))
    if not output_files:
        print(f"Skipping {anvil_id} — no output.txt found")
        continue
    output_file = output_files[0]
    with open(output_file, "r") as f:
        lines = f.readlines()
    for i, line in enumerate(lines):
        if line.startswith("//"):
            col_names = [c.strip() for c in line.lstrip("/").strip().split(";")]
            df_file = pd.read_csv(output_file, sep=";", skiprows=i + 1, header=None, names=col_names)
            df_file.columns = [c.lstrip("/") for c in df_file.columns]
            break
    if "Roughness (10)" not in df_file.columns or "condition" not in df_file.columns:
        print(f"  Skipping {anvil_id} — missing expected columns")
        continue
    subset = df_file[["Roughness (10)", "Mean curvature (10)", "condition"]].dropna()
    subset = subset.rename(columns={"Roughness (10)": "roughness", "Mean curvature (10)": "curvature"})
    subset["anvil_id"] = anvil_id
    records.append(subset)

df = pd.concat(records, ignore_index=True)

# Balance sample sizes per anvil & condition for plotting
balanced_chunks = []
for anvil in df["anvil_id"].unique():
    sub_anvil = df[df["anvil_id"] == anvil]
    n_d = (sub_anvil["condition"] == "damaged").sum()
    n_u = (sub_anvil["condition"] == "undamaged").sum()
    if n_d == 0 or n_u == 0:
        balanced_chunks.append(sub_anvil)
        continue
    target_n = min(n_d, n_u)
    damaged_sub = sub_anvil[sub_anvil["condition"] == "damaged"]
    if len(damaged_sub) > target_n:
        damaged_sub = damaged_sub.sample(n=target_n, random_state=random_seed, replace=False)
    undamaged_sub = sub_anvil[sub_anvil["condition"] == "undamaged"]
    if len(undamaged_sub) > target_n:
        undamaged_sub = undamaged_sub.sample(n=target_n, random_state=random_seed, replace=False)
    balanced_chunks.append(pd.concat([damaged_sub, undamaged_sub], ignore_index=True))

df_plot = pd.concat(balanced_chunks, ignore_index=True)

# Anvil order and grouping for plotting
group_map = {
    "KBN-AV2":      "KBN\nsandstone",
    "KBN-AV3":      "KBN\nsandstone",
    "KBN-AV4":      "KBN\nsandstone",
    "KBN-AV5":      "KBN\nsandstone",
    "KBN-AV6":      "KBN\nsandstone",
    "KBY-AV7":      "KBY\nsandstone",
    "KBY-AV8":      "KBY\nsandstone",
    "KBY-AV9":      "KBY\nsandstone",
    "KBY-AV11":     "KBY\nsandstone",
    "KBY-AV12":     "KBY\nsandstone",
    "AL-AV2":       "AL limestone\ntype I",
    "AL-AV3":       "AL limestone\ntype I",
    "AL-AV5_east":  "AL limestone\ntype I",
    "AL-AV5_west":  "AL limestone\ntype I",
    "AL-AV8":       "AL limestone\ntype I",
    "AL-AV7":       "AL limestone\ntype II",
    "AL-AV9":       "AL limestone\ntype II",
    "AL-AV12":      "AL limestone\ntype II",
    "AL-AV13":      "AL limestone\ntype II",
}

group_order = [
    "KBN\nsandstone",
    "KBY\nsandstone",
    "AL limestone\ntype I",
    "AL limestone\ntype II",
]

df_plot["group"] = df_plot["anvil_id"].map(group_map)
df_plot = df_plot[df_plot["group"].notna()]

# Print sample sizes per group and condition
print("\nSample sizes per group and condition:")
for group in group_order:
    grp = df_plot[df_plot["group"] == group]
    for condition in ["undamaged", "damaged"]:
        n = (grp["condition"] == condition).sum()
        print(f"  {group.replace(chr(10), ' '):<25} {condition}: n = {n}")

# Plot
sns.set_theme(style="whitegrid", context="talk")

fig, axes = plt.subplots(1, 4, figsize=(4 * 4, 5), sharey=True)

upper_limit = df_plot["roughness"].quantile(0.997)

for i, (group, ax) in enumerate(zip(group_order, axes)):
    grp = df_plot[df_plot["group"] == group]

    sns.boxplot(
        data=grp,
        x="condition",
        y="roughness",
        ax=ax,
        palette={"undamaged": "#1D9E75", "damaged": "#D85A30"},
        linewidth=1.5,
        flierprops={'marker': '_', 'markeredgewidth': 0.02, 'markersize': 4, 'alpha': 0.5},
        order=["undamaged", "damaged"],
    )

    ax.set_title(group, fontsize=12, weight='normal')
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_xticklabels([])
    ax.set_ylim(-0.1, upper_limit)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(i == 0)
    ax.tick_params(axis='y', left=(i == 0), length=0)

    ax.grid(axis='y', alpha=0.3, linewidth=0.5)
    ax.tick_params(axis='both', labelsize=10)

fig.text(0.04, 0.5, 'Roughness [mm]', va='center', rotation='vertical', fontsize=16)

legend_elements = [
    mpatches.Patch(facecolor='#D85A30', label='damaged'),
    mpatches.Patch(facecolor='#1D9E75', label='undamaged'),
]
fig.legend(
    handles=legend_elements,
    loc="center left",
    bbox_to_anchor=(0.92, 0.5),
    frameon=True,
    fancybox=True,
    shadow=True,
    fontsize=12,
)

plt.tight_layout(rect=[0.06, 0.02, 0.90, 0.98])
plt.show()

In [ ]:
# Curvature: boxplot by locality and raw material.

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

root_path = Path("FILEPATH")

random_seed = 4

# Load data
records = []
for anvil_dir in sorted(root_path.iterdir()):
    if not anvil_dir.is_dir():
        continue
    anvil_id = anvil_dir.name
    output_files = list(anvil_dir.glob("*_output.txt"))
    if not output_files:
        print(f"Skipping {anvil_id} — no output.txt found")
        continue
    output_file = output_files[0]
    with open(output_file, "r") as f:
        lines = f.readlines()
    for i, line in enumerate(lines):
        if line.startswith("//"):
            col_names = [c.strip() for c in line.lstrip("/").strip().split(";")]
            df_file = pd.read_csv(output_file, sep=";", skiprows=i + 1, header=None, names=col_names)
            df_file.columns = [c.lstrip("/") for c in df_file.columns]
            break
    if "Mean curvature (10)" not in df_file.columns or "condition" not in df_file.columns:
        print(f"  Skipping {anvil_id} — missing expected columns")
        continue
    subset = df_file[["Roughness (10)", "Mean curvature (10)", "condition"]].dropna()
    subset = subset.rename(columns={"Roughness (10)": "roughness", "Mean curvature (10)": "curvature"})
    subset["anvil_id"] = anvil_id
    records.append(subset)

df = pd.concat(records, ignore_index=True)

# Balance sample sizes per anvil & condition
balanced_chunks = []
for anvil in df["anvil_id"].unique():
    sub_anvil = df[df["anvil_id"] == anvil]
    n_d = (sub_anvil["condition"] == "damaged").sum()
    n_u = (sub_anvil["condition"] == "undamaged").sum()
    if n_d == 0 or n_u == 0:
        balanced_chunks.append(sub_anvil)
        continue
    target_n = min(n_d, n_u)
    damaged_sub = sub_anvil[sub_anvil["condition"] == "damaged"]
    if len(damaged_sub) > target_n:
        damaged_sub = damaged_sub.sample(n=target_n, random_state=random_seed, replace=False)
    undamaged_sub = sub_anvil[sub_anvil["condition"] == "undamaged"]
    if len(undamaged_sub) > target_n:
        undamaged_sub = undamaged_sub.sample(n=target_n, random_state=random_seed, replace=False)
    balanced_chunks.append(pd.concat([damaged_sub, undamaged_sub], ignore_index=True))

df_plot = pd.concat(balanced_chunks, ignore_index=True)

# Anvil order and grouping for plotting
group_map = {
    "KBN-AV2":      "KBN\nsandstone",
    "KBN-AV3":      "KBN\nsandstone",
    "KBN-AV4":      "KBN\nsandstone",
    "KBN-AV5":      "KBN\nsandstone",
    "KBN-AV6":      "KBN\nsandstone",
    "KBY-AV7":      "KBY\nsandstone",
    "KBY-AV8":      "KBY\nsandstone",
    "KBY-AV9":      "KBY\nsandstone",
    "KBY-AV11":     "KBY\nsandstone",
    "KBY-AV12":     "KBY\nsandstone",
    "AL-AV2":       "AL limestone\ntype I",
    "AL-AV3":       "AL limestone\ntype I",
    "AL-AV5_east":  "AL limestone\ntype I",
    "AL-AV5_west":  "AL limestone\ntype I",
    "AL-AV8":       "AL limestone\ntype I",
    "AL-AV7":       "AL limestone\ntype II",
    "AL-AV9":       "AL limestone\ntype II",
    "AL-AV12":      "AL limestone\ntype II",
    "AL-AV13":      "AL limestone\ntype II",
}

group_order = [
    "KBN\nsandstone",
    "KBY\nsandstone",
    "AL limestone\ntype I",
    "AL limestone\ntype II",
]

df_plot["group"] = df_plot["anvil_id"].map(group_map)
df_plot = df_plot[df_plot["group"].notna()]

# Print sample sizes per group and condition
print("\nSample sizes per group and condition:")
for group in group_order:
    grp = df_plot[df_plot["group"] == group]
    for condition in ["undamaged", "damaged"]:
        n = (grp["condition"] == condition).sum()
        print(f"  {group.replace(chr(10), ' '):<25} {condition}: n = {n}")

# Plot
sns.set_theme(style="whitegrid", context="talk")

fig, axes = plt.subplots(1, 4, figsize=(4 * 4, 5), sharey=True)

upper_limit = df_plot["curvature"].quantile(0.997)

for i, (group, ax) in enumerate(zip(group_order, axes)):
    grp = df_plot[df_plot["group"] == group]

    sns.boxplot(
        data=grp,
        x="condition",
        y="curvature",
        ax=ax,
        palette={"undamaged": "#1D9E75", "damaged": "#D85A30"},
        linewidth=1.5,
        flierprops={'marker': '_', 'markeredgewidth': 0.02, 'markersize': 4, 'alpha': 0.5},
        order=["undamaged", "damaged"],
    )

    ax.set_title(group, fontsize=12, weight='normal')
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_xticklabels([])
    ax.set_ylim(-0.005, upper_limit)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    if i > 0:
        ax.spines['left'].set_visible(False)
        ax.tick_params(axis='y', left=False)

    ax.grid(axis='y', alpha=0.3, linewidth=0.5)
    ax.tick_params(axis='both', labelsize=10)

fig.text(0.04, 0.5, 'Mean Curvature [mm⁻¹]', va='center', rotation='vertical', fontsize=16)

legend_elements = [
    mpatches.Patch(facecolor='#D85A30', label='damaged'),
    mpatches.Patch(facecolor='#1D9E75', label='undamaged'),
]
fig.legend(
    handles=legend_elements,
    loc="center left",
    bbox_to_anchor=(0.92, 0.5),
    frameon=True,
    fancybox=True,
    shadow=True,
    fontsize=12,
)

plt.tight_layout(rect=[0.06, 0.02, 0.90, 0.98])
plt.show()

In [ ]:
# Roughness: Dot plot of median roughness per anvil and condition

# Define custom order to group by location and raw material
anvil_order = [
    # KBN, sandstone
    "KBN-AV2", "KBN-AV3", "KBN-AV4", "KBN-AV5", "KBN-AV6",
    # KBY, sandstone
    "KBY-AV7", "KBY-AV8", "KBY-AV9", "KBY-AV11", "KBY-AV12",
    # AL - High quality limestone/type I
    "AL-AV2", "AL-AV3", "AL-AV5_east", "AL-AV5_west", "AL-AV8",
    # AL - Low quality limestone/type II
    "AL-AV7", "AL-AV9", "AL-AV12", "AL-AV13",
]

# Filter to only anvils present in data
anvils_present = [a for a in anvil_order if a in summary_df["anvil_id"].unique()]

# Minimalist theme
sns.set_theme(style="white", context="talk")

# Pivot the table
median_table = summary_df.pivot(
    index="anvil_id",
    columns="condition",
    values="median_roughness"
)

# Reindex to apply custom order
median_table = median_table.reindex(anvils_present)

fig, ax = plt.subplots(figsize=(14, 6))

ax.scatter(
    median_table.index,
    median_table["undamaged"],
    color="#1D9E75",
    s=80,
    label="undamaged",
    zorder=2,
    alpha=0.8
)

ax.scatter(
    median_table.index,
    median_table["damaged"],
    color="#D85A30",
    s=80,
    label="damaged",
    zorder=2,
    alpha=0.8
)

ax.set_ylabel("Roughness [mm]", fontsize=12)
ax.set_xlabel("Anvil", fontsize=12)
ax.tick_params(axis='both', labelsize=10)
plt.xticks(rotation=45, ha='right')
ax.set_title("Median roughness by anvil", fontsize=14, weight='normal')

# Limit the number of y ticks to 5 for better readability
ax.yaxis.set_major_locator(plt.MaxNLocator(nbins=5))
# Limit the number of decimal points in y-axis labels to 2 for better readability
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.2f}'))

# Remove top and right spines
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Lighter grid
ax.grid(axis='y', alpha=0.3, linewidth=0.5)
ax.set_axisbelow(True)

# Legend styling
ax.legend(fontsize=11, frameon=False, 
          loc='center left', bbox_to_anchor=(1, 0.5))

plt.tight_layout(rect=[0, 0, 0.95, 1])
plt.show()

In [ ]:
# Curvature: Dot plot of mediancurvature per anvil and condition

# Define custom order to group by location and raw material
anvil_order = [
    # KBN, sandstone
    "KBN-AV2", "KBN-AV3", "KBN-AV4", "KBN-AV5", "KBN-AV6",
    # KBY, sandstone
    "KBY-AV7", "KBY-AV8", "KBY-AV9", "KBY-AV11", "KBY-AV12",
    # AL - High quality limestone/type I
    "AL-AV2", "AL-AV3", "AL-AV5_east", "AL-AV5_west", "AL-AV8",
    # AL - Low quality limestone/type II
    "AL-AV7", "AL-AV9", "AL-AV12", "AL-AV13",
]

# Filter to only anvils present in data
anvils_present = [a for a in anvil_order if a in summary_df["anvil_id"].unique()]

# Minimalist theme
sns.set_theme(style="white", context="talk")

# Pivot the table
median_table = summary_df.pivot(
    index="anvil_id",
    columns="condition",
    values="median_curvature"
)

# Reindex to apply custom order
median_table = median_table.reindex(anvils_present)

fig, ax = plt.subplots(figsize=(14, 6))

ax.scatter(
    median_table.index,
    median_table["undamaged"],
    color="#1D9E75",
    s=80,
    label="undamaged",
    zorder=2,
    alpha=0.8
)

ax.scatter(
    median_table.index,
    median_table["damaged"],
    color="#D85A30",
    s=80,
    label="damaged",
    zorder=2,
    alpha=0.8
)

ax.set_ylabel("Median curvature [mm^-1]", fontsize=12)
ax.set_xlabel("Anvil", fontsize=12)
ax.tick_params(axis='both', labelsize=10)
plt.xticks(rotation=45, ha='right')
ax.set_title("Median curvature by anvil", fontsize=14, weight='normal')

# Limit the number of y ticks to 5 for better readability
ax.yaxis.set_major_locator(plt.MaxNLocator(nbins=5))
# Limit the number of decimal points in y-axis labels to 3 for better readability
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.3f}'))

# Remove top and right spines
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Lighter grid
ax.grid(axis='y', alpha=0.3, linewidth=0.5)
ax.set_axisbelow(True)

# Legend styling
ax.legend(fontsize=11, frameon=False, 
          loc='center left', bbox_to_anchor=(1, 0.5))

plt.tight_layout(rect=[0, 0, 0.95, 1])
plt.show()

In [ ]:
# Roughness: Dot plot of max roughness per anvil and condition

# Define custom order to group by location and raw material
anvil_order = [
    # KBN, sandstone
    "KBN-AV2", "KBN-AV3", "KBN-AV4", "KBN-AV5", "KBN-AV6",
    # KBY, sandstone
    "KBY-AV7", "KBY-AV8", "KBY-AV9", "KBY-AV11", "KBY-AV12",
    # AL - High quality limestone/type I
    "AL-AV2", "AL-AV3", "AL-AV5_east", "AL-AV5_west", "AL-AV8",
    # AL - Low quality limestone/type II
    "AL-AV7", "AL-AV9", "AL-AV12", "AL-AV13",
]

# Filter to only anvils present in data
anvils_present = [a for a in anvil_order if a in summary_df["anvil_id"].unique()]

# Minimalist theme
sns.set_theme(style="white", context="talk")

# Pivot the table
max_table = summary_df.pivot(
    index="anvil_id",
    columns="condition",
    values="max_roughness"
)

# Reindex to apply custom order
max_table = max_table.reindex(anvils_present)

fig, ax = plt.subplots(figsize=(14, 6))

ax.scatter(
    max_table.index,
    max_table["undamaged"],
    color="#1D9E75",
    s=80,
    label="undamaged",
    zorder=2,
    alpha=0.8
)

ax.scatter(
    max_table.index,
    max_table["damaged"],
    color="#D85A30",
    s=80,
    label="damaged",
    zorder=2,
    alpha=0.8
)

ax.set_ylabel("Roughness [mm]", fontsize=12)
ax.set_xlabel("Anvil", fontsize=12)
ax.tick_params(axis='both', labelsize=10)
plt.xticks(rotation=45, ha='right')
ax.set_title("Max roughness by anvil", fontsize=14, weight='normal')

# Limit the number of y ticks to 5 for better readability
ax.yaxis.set_major_locator(plt.MaxNLocator(nbins=5))
# Limit the number of decimal points in y-axis labels to 2 for better readability
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.2f}'))

# Remove top and right spines
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Lighter grid
ax.grid(axis='y', alpha=0.3, linewidth=0.5)
ax.set_axisbelow(True)

# Legend styling
ax.legend(fontsize=11, frameon=False, 
          loc='center left', bbox_to_anchor=(1, 0.5))

plt.tight_layout(rect=[0, 0, 0.95, 1])
plt.show()

In [ ]:
# Curvature: Dot plot of max curvature per anvil and condition

# Define custom order to group by location and raw material
anvil_order = [
    # KBN, sandstone
    "KBN-AV2", "KBN-AV3", "KBN-AV4", "KBN-AV5", "KBN-AV6",
    # KBY, sandstone
    "KBY-AV7", "KBY-AV8", "KBY-AV9", "KBY-AV11", "KBY-AV12",
    # AL - High quality limestone/type I
    "AL-AV2", "AL-AV3", "AL-AV5_east", "AL-AV5_west", "AL-AV8",
    # AL - Low quality limestone/type II
    "AL-AV7", "AL-AV9", "AL-AV12", "AL-AV13",
]

# Filter to only anvils present in data
anvils_present = [a for a in anvil_order if a in summary_df["anvil_id"].unique()]

# Minimalist theme
sns.set_theme(style="white", context="talk")

# Pivot the table
max_table = summary_df.pivot(
    index="anvil_id",
    columns="condition",
    values="max_curvature"
)

# Reindex to apply custom order
max_table = max_table.reindex(anvils_present)

fig, ax = plt.subplots(figsize=(14, 6))

ax.scatter(
    max_table.index,
    max_table["undamaged"],
    color="#1D9E75",
    s=80,
    label="undamaged",
    zorder=2,
    alpha=0.8
)

ax.scatter(
    max_table.index,
    max_table["damaged"],
    color="#D85A30",
    s=80,
    label="damaged",
    zorder=2,
    alpha=0.8
)

ax.set_ylabel("Max curvature [mm^-1]", fontsize=12)
ax.set_xlabel("Anvil", fontsize=12)
ax.tick_params(axis='both', labelsize=10)
plt.xticks(rotation=45, ha='right')
ax.set_title("Max curvature by anvil", fontsize=14, weight='normal')

# Limit the number of y ticks to 5 for better readability
ax.yaxis.set_major_locator(plt.MaxNLocator(nbins=5))
# Limit the number of decimal points in y-axis labels to 3 for better readability
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.3f}'))

# Remove top and right spines
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Lighter grid
ax.grid(axis='y', alpha=0.3, linewidth=0.5)
ax.set_axisbelow(True)

# Legend styling
ax.legend(fontsize=11, frameon=False, 
          loc='center left', bbox_to_anchor=(1, 0.5))

plt.tight_layout(rect=[0, 0, 0.95, 1])
plt.show()

In [ ]:
# CLES: Dumbbell plot for roughness and curvature data

file_path = "PATH"

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import seaborn as sns

# Load CLES data
within_df_roughness = pd.read_excel(file_path, sheet_name="Within-Anvil MW - roughness")
within_df_curvature = pd.read_excel(file_path, sheet_name="Within-Anvil MW - curvature")

# Anvil order by raw material group
anvil_order = [
    # KBN, sandstone
    "KBN-AV2", "KBN-AV3", "KBN-AV4", "KBN-AV5", "KBN-AV6",
    # KBY, sandstone
    "KBY-AV7", "KBY-AV8", "KBY-AV9", "KBY-AV11", "KBY-AV12",
    # AL - High quality limestone/type I
    "AL-AV2", "AL-AV3", "AL-AV5_east", "AL-AV5_west", "AL-AV8",
    # AL - Low quality limestone/type II
    "AL-AV7", "AL-AV9", "AL-AV12", "AL-AV13",
]
anvils = [a for a in anvil_order if a in within_df_roughness["anvil_id"].values]

# Align both dataframes to anvil order
within_df_roughness = within_df_roughness.set_index("anvil_id").reindex(anvils).reset_index()
within_df_curvature = within_df_curvature.set_index("anvil_id").reindex(anvils).reset_index()

# Viridis-matched colors
color_roughness = "#440154"  # viridis purple
color_curvature = "#21918c"  # viridis teal

sns.set_theme(style="white", context="talk")
fig, ax = plt.subplots(figsize=(16, 7))

y = np.arange(len(anvils))

ax.scatter(
    within_df_roughness["CLES"], y,
    facecolors=color_roughness, edgecolors=color_roughness,
    linewidths=1.5, s=60, zorder=3,
)
ax.scatter(
    within_df_curvature["CLES"], y,
    facecolors=color_curvature, edgecolors=color_curvature,
    linewidths=1.5, s=60, zorder=3,
)

# Reference lines
for val in [0.25, 0.5, 0.75]:
    ax.axvline(val, color="lightgrey", linewidth=1, linestyle="-", zorder=1)

# Connecting lines
gap = 0.005
for i in range(len(anvils)):
    r_val   = within_df_roughness["CLES"].iloc[i]
    c_val   = within_df_curvature["CLES"].iloc[i]
    x_left  = min(r_val, c_val) + gap
    x_right = max(r_val, c_val) - gap
    if x_right > x_left:
        ax.plot([x_left, x_right], [y[i], y[i]],
                color="black", linewidth=0.5, zorder=2)

# Axis formatting
ax.spines["left"].set_linewidth(0.75)
ax.spines["bottom"].set_linewidth(0.75)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.set_yticks(y)
ax.set_yticklabels(anvils, fontsize=10)
ax.set_xlim(0, 1.0)
ax.xaxis.set_major_locator(plt.MaxNLocator(5))
ax.set_xlabel("Common Language Effect Size", fontsize=12, labelpad=10)
ax.set_ylabel("Anvil ID", fontsize=12, labelpad=10)
ax.tick_params(axis="both", pad=8, labelsize=10)
ax.set_title("Within-anvil effect size (damaged vs. undamaged)", fontsize=14, weight="normal")

ax.grid(axis="x", alpha=0.3, linewidth=0.5)
ax.set_axisbelow(True)

# Legend outside plot area
legend_elements = [
    mpatches.Patch(facecolor=color_roughness, edgecolor=color_roughness, linewidth=1.5, label="Roughness"),
    mpatches.Patch(facecolor=color_curvature, edgecolor=color_curvature, linewidth=1.5, label="Curvature"),
]
ax.legend(
    handles=legend_elements,
    fontsize=11, frameon=False,
    loc="upper left",
    bbox_to_anchor=(1.01, 1),
    borderaxespad=0,
)

plt.tight_layout()
plt.show()

In [ ]:
# Raw material hardness: boxplot by locality as box plots.

from pathlib import Path
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

root_path = Path("PATH")

df = pd.read_excel(root_path, sheet_name="Sheet1")
df_av = df[df["ID"].str.contains("AV", na=False)].copy()

anvil_ids = [
    "KBN-AV2", "KBN-AV3", "KBN-AV4", "KBN-AV5", "KBN-AV6",
    "KBY-AV11", "KBY-AV12", "KBY-AV7", "KBY-AV8", "KBY-AV9",
    "AL-AV2", "AL-AV3", "AL-AV5-EAST", "AL-AV5-WEST", "AL-AV8",
    "AL-AV7", "AL-AV9", "AL-AV12", "AL-AV13"
]
df_av = df_av[df_av["ID"].isin(anvil_ids)].copy()

hardness_columns = [col for col in df_av.columns if "Hardness_value" in col]
df_av["avg_hardness"] = df_av[hardness_columns].mean(axis=1)

df_av["Locality"] = df_av["Island"] + " - " + df_av["RM"]

locality_filter = [
    "Koh Boi Noi - Sandstone",
    "Koh Boi Yai - Sandstone",
    "Ao Lobi - Limestone, type I",
    "Ao Lobi - Limestone, type II"
]
df_av = df_av[df_av["Locality"].isin(locality_filter)].copy()

palette = {
    "Koh Boi Noi - Sandstone":             "#1D9E75",
    "Ao Lobi - Limestone, type II":        "#9E1D47",
    "Ao Lobi - Limestone, type I":         "#1D479E",
    "Koh Boi Yai - Sandstone":             "#9E781D"
}

# Split locality strings into components for the 3-line x labels
# Format: "Island - RM" → RM on line 1, Island on line 2, n= on line 3
locality_parts = {
    "Koh Boi Noi - Sandstone":          ("Sandstone",           "Koh Boi Noi"),
    "Koh Boi Yai - Sandstone":          ("Sandstone",           "Koh Boi Yai"),
    "Ao Lobi - Limestone, type I":      ("Limestone,\ntype I",  "Ao Lobi"),
    "Ao Lobi - Limestone, type II":     ("Limestone,\ntype II", "Ao Lobi"),
}

n_counts = df_av.groupby("Locality")["ID"].count()

sns.set_theme(style="whitegrid", context="talk")
fig, ax = plt.subplots(figsize=(10, 8))  # wider

sns.boxplot(
    x="Locality", y="avg_hardness",
    data=df_av,
    palette=palette,
    order=locality_filter,
    ax=ax,
    width=0.7,        # narrower boxes for an airier feel
    linewidth=1.2,
    flierprops=dict(marker='o', markersize=5, linestyle='none')
)

# Build 3-line x-axis labels: RM / Island / n=
x_labels = []
for loc in locality_filter:
    rm, island = locality_parts[loc]
    n = n_counts.get(loc, 0)
    x_labels.append(f"{rm}\n{island}\nn={n}")

ax.set_xticklabels(x_labels, rotation=0, ha='center', linespacing=1.6, fontsize=12)  # <-- fontsize added

# Title with extra padding below it
ax.set_title(
    "Range of Anvil Hardness Values by Location and Raw Material",
    pad=20
)
ax.set_xlabel("")
ax.set_ylabel("Rockschmidt Hardness Value")
# Set y axis label font size
ax.yaxis.label.set_size(14)
# Add space between y-axis label and ticks
ax.yaxis.labelpad = 15

# Lighten the grid for an airier look
ax.yaxis.grid(True, color='grey', linewidth=0.5, alpha=0.5)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

avg_hardness_by_locality = df_av.groupby("Locality")["avg_hardness"].mean().reset_index()
print(avg_hardness_by_locality)

stats = df_av.groupby("Locality")["avg_hardness"].agg(
    mean="mean",
    min="min",
    max="max",
    median="median"
).reset_index()

for _, row in stats.iterrows():
    print(f"{row['Locality']}: mean = {row['mean']:.2f}, min = {row['min']:.2f}, max = {row['max']:.2f}, median = {row['median']:.2f}")